<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/comfyui_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 一、安装 ComfyUI 与依赖环境 (只需运行一次)
# ==========================================
import os

# 1. 切换到主目录
%cd /content

# 智能判断：防止重复克隆导致的 fatal 报错
if not os.path.exists("/content/ComfyUI"):
    print("⏳ 正在拉取 ComfyUI 官方源码...")
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    print("✨ 检测到 ComfyUI 已存在，正在拉取最新更新...")
    %cd /content/ComfyUI
    !git pull

# 确保当前路径在 ComfyUI 内
%cd /content/ComfyUI

# 2. 【核心升级】安装带有 CUDA 13.0 深度优化的 PyTorch
print("⏳ 正在安装 PyTorch 及其核心依赖 (基于 cu130)...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

# 3. 安装 ComfyUI 官方运行依赖
print("⏳ 正在安装 ComfyUI 额外依赖...")
!pip install -q -r requirements.txt

# 4. 【关键修复】独立安装 huggingface_hub 和 hf_transfer，告别警告
print("⏳ 正在配置 Hugging Face 高速下载环境...")
!pip install -q huggingface_hub hf_transfer

print("\n🎉 第一阶段完成：ComfyUI 与顶级环境依赖安装完毕！")

In [6]:
# 二、模型下载
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download

# 1. 从 Colab Secrets 中静默读取 HF Token
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("❌ 错误：请先在左侧 '🔑 Secrets' 中添加名为 'HF_TOKEN' 的密钥！")
    raise

# 2. 写入环境变量并启用 HF_TRANSFER 满速下载
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# 3. 灵活配置下载任务列表 (每个任务独立指定仓库名)
downloads = [
    # 1. Diffusion Model -> 放入 unet 文件夹
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/diffusion_models/flux2_dev_fp8mixed.safetensors",
        "local_dir": "/content/ComfyUI/models/unet"
    },
    # 2. LoRA 模型 -> 放入 loras 文件夹
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/loras/Flux2TurboComfyv2.safetensors",
        "local_dir": "/content/ComfyUI/models/loras"
    },
    # 3. Text Encoder -> 放入 clip 文件夹
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/text_encoders/mistral_3_small_flux2_bf16.safetensors",
        "local_dir": "/content/ComfyUI/models/clip"
    }

    # 💡 如果未来要下载其他人的模型，直接在下面照抄格式：
    # {
    #     "repo_id": "其他作者/其他仓库名",
    #     "filename": "文件夹/文件名.safetensors",
    #     "local_dir": "/content/ComfyUI/models/vae"
    # }
]

print(f"\n🚀 开始满速同步指定的 {len(downloads)} 个模型文件...\n")

# 4. 执行循环下载
for i, task in enumerate(downloads, 1):
    repo = task["repo_id"]
    file_path = task["filename"]
    target_dir = task["local_dir"]
    file_name = file_path.split('/')[-1]

    print(f"[{i}/{len(downloads)}] 正在从 {repo} 同步: {file_name}")
    try:
        hf_hub_download(
            repo_id=repo,
            filename=file_path,
            local_dir=target_dir,
            repo_type="model"  # 👈 已经去掉了报 warning 的参数，清爽执行
        )
        print(f"✅ 已完成并存放到: {target_dir}\n")
    except Exception as e:
        print(f"❌ 下载失败 ({file_name}): {e}\n")

print("🎉 所有指定的模型已同步完毕！界面应该没有任何黄字警告了。")


🚀 开始满速同步指定的 3 个模型文件...

[1/3] 正在从 stronman/flux2-dev 同步: flux2_dev_fp8mixed.safetensors


split_files/diffusion_models/flux2_dev_f(…):   0%|          | 0.00/35.5G [00:00<?, ?B/s]

✅ 已完成并存放到: /content/ComfyUI/models/unet

[2/3] 正在从 stronman/flux2-dev 同步: Flux2TurboComfyv2.safetensors


split_files/loras/Flux2TurboComfyv2.safe(…):   0%|          | 0.00/2.76G [00:00<?, ?B/s]

✅ 已完成并存放到: /content/ComfyUI/models/loras

[3/3] 正在从 stronman/flux2-dev 同步: mistral_3_small_flux2_bf16.safetensors
✅ 已完成并存放到: /content/ComfyUI/models/clip

🎉 所有指定的模型已同步完毕！界面应该没有任何黄字警告了。


In [ ]:
# ==========================================
# 三、内网穿透配置 (只需运行一次)
# ==========================================
import os
from google.colab import userdata

# 1. 读取 Colab Secrets
try:
    VPS_IP = userdata.get('VPS_IP')
    FRP_TOKEN = userdata.get('FRP_TOKEN')
except Exception as e:
    print("❌ 错误：无法读取 Secrets！请检查左侧是否填写并开启了 'VPS_IP' 和 'FRP_TOKEN'。")
    raise e

# 2. 下载并解压 FRP 客户端
%cd /content
if not os.path.exists("/content/frp_0.56.0_linux_amd64"):
    print("⏳ 正在下载并解压 FRP 客户端...")
    !wget -q https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz
    !tar -xzf frp_0.56.0_linux_amd64.tar.gz
    !rm frp_0.56.0_linux_amd64.tar.gz

# 3. 动态生成并写入配置文件
print("⏳ 正在生成 frpc.toml 配置文件...")
frpc_config_content = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8080
"""

with open("/content/frp_0.56.0_linux_amd64/frpc.toml", "w", encoding="utf-8") as f:
    f.write(frpc_config_content.strip())

print("✅ FRP 配置文件生成完毕！")

In [ ]:
# ==========================================
# 四、启动 FRP 和 ComfyUI (重启界面时运行)
# ==========================================
import subprocess
import threading
from google.colab import userdata

# 获取 IP 仅为了在控制台友好地打印访问链接
try:
    VPS_IP = userdata.get('VPS_IP')
except:
    VPS_IP = "你的公网IP"

# 1. 在后台新线程启动 FRP (读取刚才生成的配置文件)
def start_frpc():
    subprocess.run(["/content/frp_0.56.0_linux_amd64/frpc", "-c", "/content/frp_0.56.0_linux_amd64/frpc.toml"])

print("⏳ 正在后台唤起 FRP 穿透服务...")
threading.Thread(target=start_frpc, daemon=True).start()

print("============================================================")
print("✅ FRP 穿透已就绪！")
print(f"👉 启动完成后，请通过浏览器访问: http://{VPS_IP}:8080")
print("============================================================\n")

# 2. 启动 ComfyUI 主程序
print("⏳ 正在启动 ComfyUI 主进程...\n")
%cd /content/ComfyUI
!python main.py --dont-print-server